# Chapter 15 — Assembly: A Language Model You Can Interrogate

**Book alignment:** PyTorch From First Principles, Chapter 15

**Question this notebook isolates:** Does a tiny character-level transformer with correct next-token shift and causal attention learn the corpus (falling loss, prefix-invariant intervention, generatable text) while the unshifted identity variant collapses to a misleading near-zero loss?

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

torch.manual_seed(0)
np.random.seed(0)
torch.set_num_threads(1)
print('torch', torch.__version__)

## 1 — Tokenizer round-trip plus the next-token alignment contract

A character tokenizer must round-trip, and targets must be inputs shifted by one (`x[:,1:] == y[:,:-1]`). The unshifted variant teaches identity lookup.

In [ ]:
words = ['river', 'garden', 'market', 'shadow', 'planet', 'water', 'golden', 'hidden']
tmpl = ['That {} repeats that small {} because repeats it by the door. ', 'Near the water, that {} carries that {}. ', 'When a {} holds again, a golden {} holds. ']
rng = np.random.RandomState(0)
parts = []
for i in range(400):
    t = tmpl[i % 3]
    parts.append(t.format(rng.choice(words), rng.choice(words)))
text = ''.join(parts)
chars = sorted(set(text))
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}
V = len(chars)
enc = lambda s: [stoi[c] for c in s]
dec = lambda ids: ''.join(itos[int(i)] for i in ids)
data = torch.tensor(enc(text), dtype=torch.long)
print('vocab', V, 'chars', repr(''.join(chars)[:40]), 'len', len(data))
print('round trip:', dec(enc(text[:80])) == text[:80])

B, T = 4, 16
starts = torch.randint(0, len(data) - T - 1, (B,))
xb = torch.stack([data[i:i + T] for i in starts])
yb = torch.stack([data[i + 1:i + T + 1] for i in starts])
yb_bad = torch.stack([data[i:i + T] for i in starts])
print('aligned:', bool(torch.equal(xb[:, 1:], yb[:, :-1])))
print('misaligned==identity:', bool(torch.equal(xb, yb_bad)))

In [ ]:
assert dec(enc(text[:80])) == text[:80]
assert data.min() >= 0 and int(data.max()) == V - 1
assert torch.equal(xb[:, 1:], yb[:, :-1])
assert not torch.equal(xb[:, 1:], yb_bad[:, :-1]) or torch.equal(xb, yb_bad)
print('tokenizer + alignment verified')

## 2 — Tiny GPT trains, and causality holds by intervention

Two layers, two heads, `n_embd=32`, block 16. Correct shift must lower the loss toward structure; head split must round-trip; future edits must not move the prefix.

In [ ]:
class Attn(nn.Module):
    def __init__(self, c=32, nh=2, drop=0.0):
        super().__init__()
        self.nh = nh
        self.qkv = nn.Linear(c, 3 * c, bias=False)
        self.proj = nn.Linear(c, c, bias=False)
        self.drop = drop
    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        Dh = C // self.nh
        q = q.view(B, T, self.nh, Dh).transpose(1, 2)
        k = k.view(B, T, self.nh, Dh).transpose(1, 2)
        v = v.view(B, T, self.nh, Dh).transpose(1, 2)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=self.drop if self.training else 0.0)
        return self.proj(y.transpose(1, 2).reshape(B, T, C))

class Block(nn.Module):
    def __init__(self, c=32, nh=2):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(c), nn.LayerNorm(c)
        self.attn = Attn(c, nh)
        self.mlp = nn.Sequential(nn.Linear(c, 4 * c), nn.GELU(), nn.Linear(4 * c, c))
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        return x + self.mlp(self.ln2(x))

class TinyGPT(nn.Module):
    def __init__(self, vocab, block=16, c=32, nh=2, nl=2):
        super().__init__()
        self.block = block
        self.tok = nn.Embedding(vocab, c)
        self.pos = nn.Embedding(block, c)
        self.blocks = nn.Sequential(*[Block(c, nh) for _ in range(nl)])
        self.ln = nn.LayerNorm(c)
        self.head = nn.Linear(c, vocab, bias=False)
    def forward(self, idx):
        T = idx.shape[1]
        if T > self.block:
            raise ValueError(f'sequence length {T} exceeds block_size {self.block}')
        h = self.tok(idx) + self.pos(torch.arange(T))[None]
        return self.head(self.ln(self.blocks(h)))

torch.manual_seed(0)
gpt = TinyGPT(V)
n_params = sum(p.numel() for p in gpt.parameters())
print('params:', n_params)
qq = torch.randn(2, 6, 32)
Dh = 16
s = qq.view(2, 6, 2, Dh).transpose(1, 2)
rt = s.transpose(1, 2).reshape(2, 6, 32)
print('head round trip:', bool(torch.equal(rt, qq)))

def get_batch(B=16, T=16, shift=True):
    st = torch.randint(0, len(data) - T - 1, (B,))
    xb = torch.stack([data[i:i + T] for i in st])
    yb = torch.stack([data[i + (1 if shift else 0):i + T + (1 if shift else 0)] for i in st])
    return xb, yb

opt = torch.optim.AdamW(gpt.parameters(), lr=3e-3)
gpt.train()
xb0, yb0 = get_batch()
with torch.no_grad():
    l0 = float(F.cross_entropy(gpt(xb0).reshape(-1, V), yb0.reshape(-1)))
for step in range(150):
    xb, yb = get_batch()
    opt.zero_grad()
    F.cross_entropy(gpt(xb).reshape(-1, V), yb.reshape(-1)).backward()
    opt.step()
gpt.eval()
with torch.no_grad():
    l1 = float(F.cross_entropy(gpt(xb0).reshape(-1, V), yb0.reshape(-1)))
print(f'loss {l0:.3f} -> {l1:.3f}  ln(V)={np.log(V):.3f}')

with torch.no_grad():
    i = 4
    xa, ya = get_batch(B=2)
    xb2 = xa.clone()
    xb2[:, i + 1:] = torch.randint(0, V, (2, T - i - 1))
    la, lb = gpt(xa), gpt(xb2)
    pre = float((la[:, :i + 1] - lb[:, :i + 1]).abs().max())
    suf = float((la[:, i + 1:] - lb[:, i + 1:]).abs().max())
print(f'causal prefix delta={pre:.2e} suffix delta={suf:.2e}')

In [ ]:
assert n_params < 200000
assert bool(torch.equal(rt, qq))
assert l1 < l0 - 0.2
assert l1 < float(np.log(V))
assert pre < 1e-6
assert suf > 1e-5
print('tiny-GPT learning + causality verified')

## 3 — Generation is a separate program: greedy samples stay in-vocabulary

Feed the model's own output back token by token under `eval()`/`no_grad`, truncating to `block_size`. Misaligned training would ace its loss yet generate repetitions of the prompt.

In [ ]:
def generate(model, seed_ids, n_new=60):
    model.eval()
    idx = seed_ids.clone()
    with torch.no_grad():
        for _ in range(n_new):
            logits = model(idx[:, -model.block:])
            idx = torch.cat([idx, logits[:, -1].argmax(-1, keepdim=True)], dim=1)
    return idx

seed = data[:16].unsqueeze(0)
out = generate(gpt, seed)
sample = dec(out[0].tolist())
print('seed :', repr(dec(seed[0].tolist())))
print('sample:', repr(sample))
ids = out[0].tolist()
print('len', len(ids), 'in range', min(ids) >= 0 and max(ids) < V)

In [ ]:
assert len(ids) == 16 + 60
assert min(ids) >= 0 and max(ids) < V
assert len(set(ids)) > 3
print('generation verified')

## What we earned

A language model is a pipeline of contracts: round-tripping tokenizer, shifted targets, causal attention, shape-preserving blocks, owned parameters that move, and a generation loop distinct from training. Each is checkable on a tiny CPU model in seconds.

Appendix A compresses the whole book into a routing table: which symptom owns which instrument, and the smallest evidence packet that proves it.